# RQ1 — effort-reduction curve (TESSERA, 0%–100% expert)

**Purpose.** Builds the full additive effort-reduction curve
(`docs/research_questions.md#rq2--citizen-science-as-a-substitute-for-expert-data`):
citizen-science presences as a fixed base, expert polygons added on top
in increasing amounts, to find where added field-collection effort
stops paying for itself. All 11 points (0%, 10%, …, 100%) are computed
under one fixed-test-set methodology, so the whole curve is
comparable, and every model is scored on the pooled test presences and
on each source separately. The two endpoints were first run on their
own in
[`presence_background_citizen_science.ipynb`](presence_background_citizen_science.ipynb)
and [`expert_arm_classifiers.ipynb`](expert_arm_classifiers.ipynb),
each with its own test-block split, so those numbers are not
comparable with the ones here.

**Data.** `inat_dealbata_clean.csv` (2021–2025 window, 2 465
presences), the expert polygons, and the uniform background pool
`background_points.npz`.

**Design decisions (mine, not yet confirmed by the supervisor):**

1. **Mixing unit: polygons, not pixels.** "X% expert" = a random X% of
   the expert *polygons* (every pixel inside them), not X% of pixels
   scattered across all polygons — collecting expert data costs a
   field visit per site/patch, not per pixel, so this is the effort
   unit that actually matches "cost of going to the field."
2. **Nested sampling.** Which polygons count as "effort already spent"
   is fixed by one random order (seed 0) for the whole curve, so 20%'s
   polygons are a superset of 10%'s — the curve reads as steadily
   adding effort, not redrawing a fresh random sample at every point.
3. **One fixed test set per evaluation seed, reused at every mixing
   level.** Test blocks are drawn once per seed (not once per level)
   from the 10 km blocks that contain citizen-science or expert
   presences, and held constant across all 11 levels within that seed
   — otherwise a score change across the curve could just mean
   "different test data," not "more expert data helped." Averaged over
   5 seeds.
4. **The test presence set always includes both sources**, even at
   100% (expert-only training), and is also scored by source.

**Classifiers:** Random Forest (main) and Logistic Regression (second
classifier), presence vs. background, same configuration as the
earlier notebooks (`n_estimators=300`, `class_weight="balanced"`;
`class_weight="balanced"`, `max_iter=1000`), no re-tuning. Metrics:
ROC-AUC and the continuous Boyce index.

> **Question numbering (note of 2026-11-13).** Current numbering in
> `docs/research_questions.md`: RQ1 = effort-reduction curve (citizen
> science against expert data); RQ2 = from points to reference data
> (segmentation mask, grade and accuracy filters); RQ3 =
> transferability away from the sampled area; RQ4 = TESSERA against
> Sentinel-2.
> The text below the title was written under an earlier numbering:
> where it says "RQ2", read RQ1. The design-doc anchor cited as
> `#rq2--citizen-science-as-a-substitute-for-expert-data` is now
> `#rq1--citizen-science-as-a-substitute-for-expert-data`.

## 1. Load pilot embeddings, background pool, citizen-science and expert presences

Expert presence pixels are recovered the same way as
`expert_arm_classifiers.ipynb` (spatial join against
`expert_polygons_clean.gpkg`, since there is no precomputed `pixel`
column for polygons), but this time also keeping, for every matched
pixel, **which polygon it came from** (`index_right` from the spatial
join) — needed for the polygon-level mixing in Section 2. The one
known overlapping polygon pair (1.62 m², see `expert_polygons.ipynb`)
can match a pixel to two polygons; duplicates are dropped, keeping the
first match, so every pixel counts once.


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.background import load_background
from src.embeddings import block_id, load_pilot_embeddings, match_to_pixel

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"
BG_FILE = EMB_DIR / "tessera_v1.1_2023_rq1_background" / "background_points.npz"

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)
# Built in notebooks/05_data_integration/background_pools.ipynb; load_background() checks the
# file and its inputs against data/processed_manifest.csv before loading.
bg = load_background(ROOT, "uniform")
bg_x, bg_y, bg_emb = bg["x"], bg["y"], bg["emb"]

# citizen-science presences
dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
cs_x, cs_y, cs_emb, cs_n_matched = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)

# expert presences, pixel -> polygon id
expert_gdf = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg").to_crs(32629)
pilot_points = gpd.GeoDataFrame(pilot_df, geometry=gpd.points_from_xy(pilot_df["x"], pilot_df["y"]), crs=32629)
joined = gpd.sjoin(pilot_points, expert_gdf[["geometry"]], predicate="within", how="inner")
joined = joined[~joined.index.duplicated(keep="first")]
expert_idx = joined.index.values
expert_poly_id_all = joined["index_right"].values
expert_emb_all = pilot_emb[expert_idx]
valid = ~np.isnan(expert_emb_all).any(axis=1)
expert_x, expert_y = pilot_df["x"].values[expert_idx][valid], pilot_df["y"].values[expert_idx][valid]
expert_emb = expert_emb_all[valid]
expert_poly_id = expert_poly_id_all[valid]

cs_block = block_id(cs_x, cs_y)
expert_block = block_id(expert_x, expert_y)
bg_block = block_id(bg_x, bg_y)

print(f"citizen-science presences: {len(cs_emb):,} (matched {cs_n_matched}/{len(dealbata)})")
print(f"expert presence pixels: {len(expert_emb):,}, across {len(np.unique(expert_poly_id)):,} polygons")
print(f"background points: {len(bg_emb):,}")


## 2. Effort order and mixing levels

A fixed random order over the expert polygons (seed 0, independent of
the 5 evaluation seeds in Section 3) decides which polygons "count" as
effort already spent at each level — this order is reused for every
evaluation seed, so the x-axis always means the same subset of
polygons, only the test/train split underneath it changes. Reported
`total_N` is nominal (full dataset, not one train fold) — the citizen-
science base (fixed) plus the expert pixels belonging to the included
polygons at that level, matching `docs/research_questions.md`'s
"report total N alongside the mixing percentage."


In [ ]:
LEVELS = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

poly_ids_available = np.unique(expert_poly_id)
poly_order = np.random.default_rng(0).permutation(poly_ids_available)

def included_polygons(level_pct):
    if level_pct >= 100:
        return poly_ids_available
    n_incl = round(level_pct / 100 * len(poly_order))
    return poly_order[:n_incl]

nominal_rows = []
for lvl in LEVELS:
    incl = included_polygons(lvl)
    n_expert_px = len(expert_poly_id) if lvl >= 100 else int(np.isin(expert_poly_id, incl).sum())
    n_cs = 0 if lvl >= 100 else len(cs_emb)
    nominal_rows.append({
        "level_pct": lvl, "n_polygons_included": len(incl),
        "n_expert_pixels": n_expert_px, "n_cs_presences": n_cs, "total_N": n_cs + n_expert_px,
    })
nominal_df = pd.DataFrame(nominal_rows)
nominal_df


## 3. Fixed spatial-block test set per seed

Test blocks are drawn once per seed from the blocks containing either
citizen-science or expert presences, and reused at every mixing level
within that seed (see "Design decisions" above). 20% of candidate
blocks held out, 5 seeds.


In [ ]:
candidate_blocks = sorted(set(cs_block) | set(expert_block))
SEEDS = range(1, 6)
test_blocks_by_seed = {}
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    test_blocks_by_seed[seed] = set(
        rng.choice(candidate_blocks, size=max(1, round(0.2 * len(candidate_blocks))), replace=False)
    )

print(f"{len(candidate_blocks)} candidate blocks (citizen-science ∪ expert presences)")
for seed, tb in test_blocks_by_seed.items():
    print(f"seed {seed}: {len(tb)} test blocks")


## 4. Run the curve and score it by source of the test presences

For each seed and level, Random Forest and Logistic Regression are
trained once and scored three times against the same test background:
on all test presences (pooled), on the expert ones only and on the
citizen-science ones only. At 100% the citizen-science base is removed
and training is expert-only.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

from src.evaluation import continuous_boyce

# The two presence-vs-background configurations.
check_configs = [
    ("RandomForest", lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
    ("LogisticRegression", lambda: LogisticRegression(class_weight="balanced", max_iter=1000)),
]
CHECK_LEVELS = LEVELS  # all 11 levels, one run

source_records, test_size_rows = [], []
for seed in SEEDS:
    test_blocks = test_blocks_by_seed[seed]
    cs_test_mask = np.isin(cs_block, list(test_blocks))
    expert_test_mask = np.isin(expert_block, list(test_blocks))
    bg_test_mask = np.isin(bg_block, list(test_blocks))

    cs_train_emb, cs_test_emb = cs_emb[~cs_test_mask], cs_emb[cs_test_mask]
    expert_train_emb, expert_test_emb = expert_emb[~expert_test_mask], expert_emb[expert_test_mask]
    expert_train_poly = expert_poly_id[~expert_test_mask]
    bg_train_emb, bg_test_emb = bg_emb[~bg_test_mask], bg_emb[bg_test_mask]

    test_size_rows.append({
        "seed": seed,
        "n_test_cs": len(cs_test_emb),
        "n_test_expert_pixels": len(expert_test_emb),
        "n_test_expert_polygons": len(np.unique(expert_poly_id[expert_test_mask])),
        "n_test_expert_blocks": len(np.unique(expert_block[expert_test_mask])),
        "expert_share_of_test_presences": len(expert_test_emb) / (len(cs_test_emb) + len(expert_test_emb)),
        "n_test_background": len(bg_test_emb),
    })

    for lvl in CHECK_LEVELS:
        if lvl >= 100:  # expert only, no citizen science
            train_pres_emb = expert_train_emb
        else:
            incl = included_polygons(lvl)
            train_pres_emb = np.concatenate([cs_train_emb, expert_train_emb[np.isin(expert_train_poly, incl)]])
        X_train = np.concatenate([train_pres_emb, bg_train_emb])
        y_train = np.concatenate([np.ones(len(train_pres_emb)), np.zeros(len(bg_train_emb))])
        scaler = StandardScaler().fit(X_train)

        for name, ctor in check_configs:
            model = ctor().fit(scaler.transform(X_train), y_train)
            score_cs = model.predict_proba(scaler.transform(cs_test_emb))[:, 1]
            score_expert = model.predict_proba(scaler.transform(expert_test_emb))[:, 1]
            score_bg = model.predict_proba(scaler.transform(bg_test_emb))[:, 1]
            # Each subset of test presences is scored against the same test background.
            subsets = {
                "pooled": np.concatenate([score_cs, score_expert]),
                "expert only": score_expert,
                "citizen science only": score_cs,
            }
            for subset, score_pres in subsets.items():
                y_true = np.concatenate([np.ones(len(score_pres)), np.zeros(len(score_bg))])
                source_records.append({
                    "seed": seed, "level_pct": lvl, "classifier": name, "test_presences": subset,
                    "n_test_presences": len(score_pres),
                    "roc_auc": roc_auc_score(y_true, np.concatenate([score_pres, score_bg])),
                    "boyce": continuous_boyce(score_pres, score_bg),
                })

test_size_df = pd.DataFrame(test_size_rows)
source_df = pd.DataFrame(source_records)
source_summary = (
    source_df.groupby(["classifier", "level_pct", "test_presences"], sort=False)
    .agg(roc_auc_mean=("roc_auc", "mean"), roc_auc_sd=("roc_auc", "std"),
         boyce_mean=("boyce", "mean"), boyce_sd=("boyce", "std"),
         n_test_presences_mean=("n_test_presences", "mean"))
    .reset_index()
)

print("Test presences per seed:")
display(test_size_df.round(3))
for metric in ["roc_auc", "boyce"]:
    print(f"{metric}, mean over 5 seeds, by level (% expert polygons):")
    display(source_summary.pivot_table(index=["classifier", "test_presences"], columns="level_pct",
                                       values=f"{metric}_mean", sort=False).round(3))

## 5. Reuse of expert test blocks across seeds

The seeds draw their test blocks independently, so the same blocks and pixels can enter the test set of more than one seed.

In [ ]:
# Are the seeds independent? Expert test blocks and pixels reused across seeds.
from collections import Counter

expert_blocks_all = set(np.unique(expert_block))
pixels_in_block = Counter(expert_block.tolist())
block_uses, pixel_sets, reuse_rows = Counter(), [], []
for seed in SEEDS:
    blocks = sorted(str(b) for b in set(test_blocks_by_seed[seed]) & expert_blocks_all)
    block_uses.update(blocks)
    pixel_sets.append(set(np.flatnonzero(np.isin(expert_block, blocks))))
    counts = [pixels_in_block[b] for b in blocks]
    reuse_rows.append({
        "seed": seed, "expert_test_blocks": ", ".join(blocks),
        "expert_pixels_per_block": ", ".join(map(str, counts)),
        "largest_block_share": max(counts) / sum(counts),
    })

display(pd.DataFrame(reuse_rows).round(3))
print(f"{len(block_uses)} distinct blocks with expert polygons enter a test set, of {len(expert_blocks_all)}; "
      f"{sum(v >= 2 for v in block_uses.values())} of them in two seeds or more")
print(f"{len(set().union(*pixel_sets)):,} distinct expert test pixels in {sum(map(len, pixel_sets)):,} summed over the seeds")

## 6. Paired differences per seed, and saved run

Every level is compared with 0% inside each seed (same test blocks). Tables, figure and the checksums of the input files go to `results/rq2_per_source_check/`.

In [ ]:
# Paired differences per seed (same test blocks at every level), and one saved copy of this run.
import hashlib
import json
from datetime import date

wide = source_df.pivot_table(index=["classifier", "test_presences", "seed"], columns="level_pct", values=["roc_auc", "boyce"])
paired_rows = []
for metric in ["roc_auc", "boyce"]:
    for lvl in CHECK_LEVELS[1:]:
        diff = (wide[metric][lvl] - wide[metric][0]).rename("diff").reset_index()
        for (clf, subset), d in diff.groupby(["classifier", "test_presences"], sort=False):
            paired_rows.append({
                "classifier": clf, "test_presences": subset, "metric": metric, "comparison": f"{lvl}% minus 0%",
                "diff_mean": d["diff"].mean(), "diff_sd": d["diff"].std(),
                "seeds_up": int((d["diff"] > 0).sum()), "seeds_down": int((d["diff"] < 0).sum()),
            })
paired_df = pd.DataFrame(paired_rows)
display(paired_df[paired_df["comparison"] == "90% minus 0%"].round(3))

OUT_DIR = ROOT / "results" / "rq2_per_source_check"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CS_FILE = ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv"
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
source_df.to_csv(OUT_DIR / "per_seed.csv", index=False)
source_summary.to_csv(OUT_DIR / "summary.csv", index=False)
paired_df.to_csv(OUT_DIR / "paired_differences.csv", index=False)
test_size_df.to_csv(OUT_DIR / "test_sizes.csv", index=False)
pd.DataFrame(reuse_rows).to_csv(OUT_DIR / "block_reuse.csv", index=False)
(OUT_DIR / "run.json").write_text(json.dumps({
    "run_date": date.today().isoformat(), "notebook": "notebooks/07_foundation_models/effort_reduction_curve.ipynb",
    "levels_pct": CHECK_LEVELS, "seeds": list(SEEDS), "classifiers": [n for n, _ in check_configs],
    "embeddings": PILOT_DIR.name, "n_cs_presences": len(cs_emb), "n_expert_pixels": len(expert_emb),
    "n_background": len(bg_emb), "n_candidate_blocks": len(candidate_blocks),
    "presence_file_sha256": sha(CS_FILE), "background_file_sha256": sha(BG_FILE),
}, indent=1))
print(f"saved to {OUT_DIR.relative_to(ROOT)}")

## 7. Plot: ROC-AUC by level and test subset

In [ ]:
# Curve by test subset: small multiples with one shared y-axis. Colour = classifier.
import matplotlib.pyplot as plt

PLOT_COLORS = {"RandomForest": "#eb6834", "LogisticRegression": "#1baf7a"}
PLOT_MARKERS = {"RandomForest": "s", "LogisticRegression": "^"}
PANELS = [("pooled", "Pooled test presences"), ("expert only", "Expert presences only"),
          ("citizen science only", "Citizen-science presences only")]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for ax, (subset, title) in zip(axes, PANELS):
    for clf, color in PLOT_COLORS.items():
        sub = source_summary[(source_summary["classifier"] == clf) & (source_summary["test_presences"] == subset)].sort_values("level_pct")
        ax.plot(sub["level_pct"], sub["roc_auc_mean"], color=color, marker=PLOT_MARKERS[clf], markersize=6, linewidth=2, label=clf)
        ax.fill_between(sub["level_pct"], sub["roc_auc_mean"] - sub["roc_auc_sd"], sub["roc_auc_mean"] + sub["roc_auc_sd"],
                        color=color, alpha=0.12)
    ax.set_title(title)
    ax.set_xlabel("% of expert polygons in training (100% = expert only)")
    ax.set_xticks(LEVELS)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8, zorder=0)
axes[0].set_ylabel("ROC-AUC (mean ± sd over 5 seeds)")
axes[0].legend(frameon=False, loc="lower left")
fig.suptitle("Effort-reduction curve by test subset — TESSERA, 2021–2025 presences, current background pool", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "curve_by_source.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Reading the curve

- **Size of the expert test subset:** 103 to 589 expert pixels per
  seed (mean 424), 40% to 70% of the test presences in each seed
  (2 122 of 4 668, 45%, summed over the five seeds).
- **The independent units are polygons and 10 km blocks, not pixels,
  and seeds reuse blocks.** Each seed tests on 2 to 4 blocks with
  expert polygons. Over the five seeds only 9 of the 18 such blocks
  enter a test set, 6 of them in two seeds; the 2 122 expert test
  pixels are 1 265 distinct ones. In seeds 1 and 3 a single block
  gives 472 of the 523 and of the 557 expert test pixels. Agreement
  between seeds is therefore not independent evidence.
- **The 0% model does not score the same on the two sources.** Random
  Forest trained on citizen science only: ROC-AUC 0.976 on expert
  presences, 0.812 on citizen-science presences, 0.902 pooled (means;
  the expert value is the higher one in every seed).
- **The citizen-science-only model ranks the expert-mapped sites
  well** (0.976 for Random Forest, 0.927 for Logistic Regression).
  The higher value on expert presences may mean they are easier to
  separate from the background, not that the two sources are
  equivalent.
- **Random Forest, pooled:** 0.902 at 0%, then 0.906 to 0.908 at every
  level from 10% to 90%. The paired difference to 0% is about +0.005
  throughout; it is larger than its sd and positive in all five seeds
  at 20% to 50% (e.g. +0.005, sd 0.003, at 20%), and not at 60% to 90%
  (at 90%: +0.004, sd 0.005, four seeds up). A small step at the first
  10%, then flat.
- **Random Forest, expert presences only:** 0.976 → 0.986 (10%) →
  0.993 (90%); paired difference at 90% +0.017 (sd 0.005), all five
  seeds, and already +0.010 at 10%.
- **Random Forest, citizen-science presences only:** 0.812 → 0.801 at
  90% (−0.011, sd 0.013).
- **Logistic Regression gains more:** pooled 0.893 → 0.910 (10%) →
  0.924 (90%), +0.031 (sd 0.007); expert only 0.927 → 0.987, +0.060
  (sd 0.012); all five seeds in both.
- **The Boyce index moves the other way on expert presences** (Random
  Forest: 0.970 at 0%, 0.940 at 90%; lower in three seeds, equal in
  two; between 0.949 and 0.963 at the levels in between). With 10
  windows it takes few distinct values (0.969 in four seeds at 0%).
- **Expert-only training (100%)** keeps the expert subset high (Random
  Forest ROC-AUC 0.994, Boyce 0.961) and loses on citizen-science
  presences (0.677, sd 0.096; Boyce 0.585, sd 0.436). Pooled: 0.854.
  The drop at 100% in the curve comes from the citizen-science
  presences.

**Caveats:** ROC-AUC against background points is not an accuracy;
each subset is scored against the same test background, so the rows
of one model differ only in the presences.

**Background pool used.** `background_points.npz` as rebuilt on
2026-11-13 by `one_class_method_selection.ipynb`: same code, seed 42
and 2 000 points, with the 50 m exclusion buffer now drawn around the
2 465 presences of 2021–2025 (checked: the file is reproduced exactly
from that seed and the current presence file). The runs stored in
`results/temporal_window_2021_2025/` used the previous pool, built
around the 3 000 all-years presences, which no longer exists. Between
the two pools the pooled Random Forest ROC-AUC agrees to 0.002 (0.902
here vs. 0.903 at 0%, 0.854 vs. 0.855 at 100%), but the pooled Boyce
index at 100% is 0.969 here against 0.930 there.